In [1]:
import numpy as np

In [4]:
from sklearn.tree import DecisionTreeRegressor

class BoostingRegressor:

    def __init__(self, n_estimators=50, learning_rate=0.1):
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.models = []
        self.initial_prediction = None

    def fit(self, X, y):

        X = np.array(X)
        y = np.array(y)

        # Initial prediction (mean)
        self.initial_prediction = np.mean(y)

        # Initialize predictions
        predictions = np.full(len(y), self.initial_prediction)

        self.models = []

        for _ in range(self.n_estimators):

            # Calculate residuals
            residuals = y - predictions

            # Train weak learner on residuals
            model = DecisionTreeRegressor(max_depth=3)

            model.fit(X, residuals)

            # Predict residuals
            residual_predictions = model.predict(X)

            # Update predictions
            predictions += self.learning_rate * residual_predictions

            # Store model
            self.models.append(model)

        return self

    def predict(self, X):

        X = np.array(X)

        predictions = np.full(
            X.shape[0],
            self.initial_prediction
        )

        for model in self.models:

            predictions += self.learning_rate * model.predict(X)

        return predictions

In [3]:
import pandas as pd

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split

data = load_diabetes(as_frame=True)

df = data.frame

df.head()

,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6,target
0,0.038076,0.050680,0.061696,0.021872,-0.044223,-0.034821,-0.043401,-0.002592,0.019907,-0.017646,151.0
1,-0.001882,-0.044642,-0.051474,-0.026328,-0.008449,-0.019163,0.074412,-0.039493,-0.068332,-0.092204,75.0
2,0.085299,0.050680,0.044451,-0.005670,-0.045599,-0.034194,-0.032356,-0.002592,0.002861,-0.025930,141.0
3,-0.089063,-0.044642,-0.011595,-0.036656,0.012191,0.024991,-0.036038,0.034309,0.022688,-0.009362,206.0
4,0.005383,-0.044642,-0.036385,0.021872,0.003935,0.015596,0.008142,-0.002592,-0.031988,-0.046641,135.0


In [5]:
X = df.drop("target", axis=1)
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

X_train: (353, 10)
X_test: (89, 10)


In [6]:
boosting_reg = BoostingRegressor(
    n_estimators=50,
    learning_rate=0.1
)

boosting_reg.fit(X_train, y_train)

y_pred = boosting_reg.predict(X_test)

In [7]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

print("MAE:", mean_absolute_error(y_test, y_pred))

print("MSE:", mean_squared_error(y_test, y_pred))

print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))

print("R2 Score:", r2_score(y_test, y_pred))

MAE: 43.91267815170555
MSE: 2829.4354425103224
RMSE: 53.192437831991896
R2 Score: 0.4659580214359622
